In [1]:
import pandas as pd
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
from tqdm import tqdm
import re

df = pd.read_csv("../../data/processed/hotel_general_info_cleaned.csv")

In [2]:
def clean_address_final(row):
    address = row['hotel_address']
    region = row['region']
    
    if not isinstance(address, str):
        return ""
        
    # 1. Sửa lỗi chính tả/encoding tiếng Việt
    replacements = {
        r'B nh': 'Bình', r'Th nh': 'Thành', r'H a': 'Hòa',
        r'Ch Minh': 'Chí Minh', r'c ng': 'công', r'L Duẩn': 'Lê Duẩn',
        r'H ng Vương': 'Hùng Vương', r'Khu ph ': 'Khu phố ',
        r'Qu n ': 'Quận '
    }
    
    temp_addr = address
    for bad, good in replacements.items():
        temp_addr = re.sub(bad, good, temp_addr, flags=re.IGNORECASE)

    parts = [p.strip() for p in temp_addr.split(',')]
    cleaned_parts = []
    seen_normalized = set()
    
    # Danh sách TP dễ bị lẫn
    conflicting_cities = ['hồ chí minh', 'hà nội', 'đà nẵng', 'cần thơ', 'hải phòng']
    region_norm = str(region).lower().strip() if pd.notna(region) else ""

    # Duyệt ngược từ cuối lên đầu (để ưu tiên giữ Tỉnh/Thành phố chuẩn nhất ở cuối)
    for i, part in enumerate(reversed(parts)):
        p_clean = part.strip()
        if not p_clean: continue
            
        # Xóa Zipcode và SĐT
        if re.match(r'^\d{5,6}$', p_clean): continue
        if re.match(r'^\+?\d{8,15}$', p_clean): continue
        
        # Xóa số rác (1-2 chữ số) trừ khi nó là số nhà (phần tử đầu tiên của chuỗi gốc)
        is_first_part_of_original = (i == len(parts) - 1)
        if re.match(r'^\d{1,2}$', p_clean) and not is_first_part_of_original:
            continue

        # Xóa Vietnam để thêm lại sau cho chuẩn
        if p_clean.lower() in ['việt nam', 'vietnam', 'viet nam', 'vn']: continue
        
        # Kiểm tra mâu thuẫn Region
        p_lower = p_clean.lower()
        has_conflict = False
        for city in conflicting_cities:
            if city in p_lower and city not in region_norm:
                 # Nếu tên TP lạ xuất hiện mà không phải tên đường -> XÓA
                 if "đường" not in p_lower and "calle" not in p_lower and "street" not in p_lower:
                     has_conflict = True
                     break
        if has_conflict: continue

        # Khử trùng lặp
        p_norm_core = re.sub(r'^(tỉnh|thành phố|tp\.?|quận|huyện|thị xã|phường|xã)\s+', '', p_lower).strip()
        is_duplicate = False
        for seen in seen_normalized:
            if p_norm_core == seen: # Đã có "Bình Dương" thì bỏ "Tỉnh Bình Dương"
                is_duplicate = True
                break
        
        if is_duplicate: continue
            
        cleaned_parts.append(p_clean)
        seen_normalized.add(p_norm_core)

    cleaned_parts.reverse()
    final_address = ", ".join(cleaned_parts)
    if final_address: final_address += ", Việt Nam"
    return final_address

In [3]:
# 3. Áp dụng làm sạch
print("Đang làm sạch dữ liệu...")
df = df[:10].copy()
df['cleaned_address'] = df.apply(clean_address_final, axis=1)

# 4. Cấu hình Geopy
# QUAN TRỌNG: Thay 'my_hotel_project_email@gmail.com' bằng tên bất kỳ của bạn
geolocator = Nominatim(user_agent="Vanh_data_mining@gmail.com")

# RateLimiter giúp tự động delay 1 giây giữa các request để không bị lỗi 429 Too Many Requests
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1.1)

# 5. Chạy Geocoding (Có thanh tiến trình)
print("Đang lấy toạ độ (việc này sẽ tốn thời gian)...")
tqdm.pandas() # Khởi tạo tqdm cho pandas

# Gọi hàm geocode lên cột đã clean
# timeout=10 để tránh treo nếu mạng lag
df['location'] = df['cleaned_address'].progress_apply(lambda addr: geocode(addr, timeout=10) if addr else None)

# 6. Tách cột latitude và longitude
df['latitude'] = df['location'].apply(lambda loc: loc.latitude if loc else None)
df['longitude'] = df['location'].apply(lambda loc: loc.longitude if loc else None)

# Xóa cột trung gian nếu muốn cho gọn
# df = df.drop(columns=['location'])

# 7. Lưu file kết quả
df.to_csv('hotel_general_info_with_coords.csv', index=False)
print("Hoàn tất! Đã lưu file hotel_general_info_with_coords.csv")

# Kiểm tra tỷ lệ thành công
success_rate = df['latitude'].notnull().mean() * 100
print(f"Tỷ lệ tìm thấy toạ độ: {success_rate:.2f}%")

Đang làm sạch dữ liệu...
Đang lấy toạ độ (việc này sẽ tốn thời gian)...


100%|██████████| 10/10 [00:12<00:00,  1.22s/it]

Hoàn tất! Đã lưu file hotel_general_info_with_coords.csv
Tỷ lệ tìm thấy toạ độ: 20.00%
